# Building Chat Ui's with Gradio: Your First Conversational AI Assistant

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
API_KEY = os.getenv("API_KEY")
API_KEY[:5]

'sk-pr'

In [5]:
openai = OpenAI(api_key=API_KEY)

In [6]:
MODEL = "gpt-4.1-mini"

In [7]:
system_message= "You are helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [8]:
def chat(message,history):
    return "bananas"

In [9]:
chat('f','g')

'bananas'

In [11]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [12]:
def chat(message,history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [13]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [14]:
# Building a Streaming Chatbot with Gradio and OpenAi API

In [21]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [29]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}]+history+[{"role":"user","content":message}]

    response = openai.chat.completions.create(model=MODEL,messages=messages)

    return response.choices[0].message.content

In [25]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [30]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [34]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages,stream=True)

    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ""
        yield response


In [35]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


# System Prompt, MultiShot promting and Your First look at RAG

In [39]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [40]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [46]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


In [43]:
def chat(message, history):
    relevant_system_message = system_message
    if "belt" in message.lower():
        relevant_system_message+=" The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages,stream=True)

    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ""
        yield response

In [44]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


In [45]:
# selecting relavant context